In [1]:
import pandas as pd
import numpy as np
import pickle

import sys
sys.path.insert(0, '..')

from utilities_notebook import zscore_normalize_features, apply_zscore_normalization, get_engineered_data
import yfinance as yf

from datetime import datetime
from zoneinfo import ZoneInfo

# Load your model to check the 'expected_features'
with open('../model.pkl', 'rb') as f:
    model = pickle.load(f)

In [2]:
model

{'weights': array([ 0.10420549, -0.52079175, -0.41705808,  0.12061994,  0.24286792,
         0.44700312,  0.03788581, -0.06492184,  0.19082469,  0.53794669,
         0.32588546,  0.07008246,  0.15135067,  0.22252467, -0.06476562,
         0.37409437,  0.06320299,  0.01156817,  0.0409008 ,  0.05777237,
         0.11835459, -0.30667039, -0.11492176,  0.12208219, -0.1908274 ,
        -0.78291617, -0.12055786]),
 'bias': np.float64(-1.4475714946980056),
 'mu': volume                  9.201526e+06
 pct_change_1d           3.303557e-04
 rsi                     4.938905e+01
 adx                     3.095551e+01
 volume^2                1.139406e+14
 pct_change_1d^2         3.340574e-04
 rsi^2                   2.610649e+03
 adx^2                   1.103528e+03
 volatility^2            4.175331e+00
 volume*pct_change_1d    1.500027e+04
 volume*rsi              4.684029e+08
 volume*adx              2.940709e+08
 volume*corr             3.600314e+06
 volume*volatility       1.814237e+07
 pct_cha

In [3]:
import numpy as np
ALL_TRAINING_FEATURES = [
    'volume', 'pct_change_1d', 'rsi', 'adx', 'corr', 'volatility',
    'volume^2', 'pct_change_1d^2', 'rsi^2', 'adx^2', 'corr^2', 'volatility^2',
    'volume*pct_change_1d', 'volume*rsi', 'volume*adx', 'volume*corr', 
    'volume*volatility', 'pct_change_1d*rsi', 'pct_change_1d*adx', 
    'pct_change_1d*corr', 'pct_change_1d*volatility', 'rsi*adx', 'rsi*corr', 
    'rsi*volatility', 'adx*corr', 'adx*volatility', 'corr*volatility'
]

In [4]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    f_wb = 1 / (1 + np.exp(-z))

    return f_wb

In [5]:


data = yf.download('5296.KL', period='30d', multi_level_index=False)
data

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2026-01-07,1.56,1.56,1.50,1.52,12231500
2026-01-08,1.54,1.55,1.53,1.55,9367900
2026-01-09,1.58,1.59,1.53,1.55,17083700
2026-01-12,1.63,1.63,1.56,1.58,21047500
2026-01-13,1.67,1.69,1.62,1.63,20931300
2026-01-14,1.73,1.73,1.65,1.67,22948000
2026-01-15,1.72,1.74,1.68,1.73,20878000
2026-01-16,1.73,1.73,1.70,1.72,21296100
2026-01-19,1.73,1.73,1.73,1.73,0


In [6]:
def is_market_ticking(ticker_symbol):
    ticker = yf.Ticker(ticker_symbol)
    
    state = ticker.info.get('marketState')

    if state == 'REGULAR' or state == 'PRE':
        return True
    else:
        return False

In [7]:
is_market_ticking('5296.KL')

False

In [8]:
tick = yf.Ticker('5296.KL')
dat = tick.history(period='2d', interval='4h')
dat2 = yf.download(tickers='5296.KL', period='5d')
dat2

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,5296.KL,5296.KL,5296.KL,5296.KL,5296.KL
Date,,,,,
2026-02-12,1.84,1.86,1.81,1.85,11216600
2026-02-13,1.85,1.86,1.82,1.84,10172400
2026-02-16,1.84,1.88,1.83,1.85,6330000
2026-02-19,1.87,1.89,1.84,1.84,6376600
2026-02-20,1.87,1.88,1.84,1.87,5870800
